# Forecasting volume penumpang

Membandingkan seasonal-naive dengan ETS additive pada holdout 2025, memilih metode sederhana secara eksplisit, lalu membuat forecast tiga bulan.

Semua input dan output saat ini adalah prototipe sintetis. Hasil tidak boleh dianggap sebagai observasi lapangan atau rekomendasi bisnis/investasi produksi.

In [1]:
from pathlib import Path

def find_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'ml' / 'DATASET_CATALOG.md').exists():
            return candidate
    raise FileNotFoundError('Jalankan notebook dari dalam repository TCI')

ROOT = find_root()
ML_ROOT = ROOT / 'ml'
SEED = 20260911
STATUS = 'synthetic_prototype'
print(f'Project root: {ROOT}')

Project root: C:\Users\axels\Axel Documents\Documents\BINUS\Lomba\MAPID WebGIS (Top 50)\App\TCI


In [2]:

import json
import pickle
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing

feature_dir = ML_ROOT / "forecasting"
data_path = feature_dir / "data" / "passenger_volume_monthly.csv"
crosswalk_path = ML_ROOT / "shared" / "processed" / "station_crosswalk.csv"
models_dir, outputs_dir = feature_dir / "models", feature_dir / "outputs"
models_dir.mkdir(parents=True, exist_ok=True)
outputs_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(data_path, dtype={"station_code": str, "station_id": str})
crosswalk = pd.read_csv(crosswalk_path)
df["period_date"] = pd.to_datetime(df["period"] + "-01")
df = df.sort_values(["station_code", "period_date"]).reset_index(drop=True)

duplicate_count = int(df.duplicated(["station_code", "period"]).sum())
negative_count = int((df["passenger_volume_monthly"] < 0).sum())
missing_months = {}
for code, group in df.groupby("station_code"):
    expected = pd.date_range(group.period_date.min(), group.period_date.max(), freq="MS")
    missing = expected.difference(group.period_date)
    if len(missing):
        missing_months[code] = [value.strftime("%Y-%m") for value in missing]
invalid_coordinates = int((~crosswalk.latitude.between(-7.0, -5.8) | ~crosswalk.longitude.between(105.8, 107.5)).sum())
crosswalk_mismatch = int((set(df.station_code) ^ set(crosswalk.station_code)).__len__())
quality_passed = not (duplicate_count or negative_count or missing_months or invalid_coordinates or crosswalk_mismatch)
quality_report = {
    "passed": quality_passed,
    "record_count": len(df),
    "station_count": int(df.station_code.nunique()),
    "period_min": df.period.min(), "period_max": df.period.max(),
    "duplicate_station_periods": duplicate_count,
    "negative_volumes": negative_count,
    "missing_months_by_station": missing_months,
    "invalid_coordinates": invalid_coordinates,
    "crosswalk_mismatch_count": crosswalk_mismatch,
    "imputed_records": 0,
}
(outputs_dir / "data_quality_report.json").write_text(json.dumps(quality_report, indent=2), encoding="utf-8")
if not quality_passed:
    raise ValueError("Forecasting quality gate failed; outputs are unavailable")

def seasonal_naive(train_values, horizon):
    values = list(map(float, train_values))
    result = []
    for _ in range(horizon):
        value = values[-12] if len(values) >= 12 else values[-1]
        result.append(value)
        values.append(value)
    return np.asarray(result)

station_metrics, validation_predictions = [], []
ets_wins = 0
for code, group in df.groupby("station_code", sort=True):
    train = group[group.period_date < "2025-01-01"]
    test = group[group.period_date >= "2025-01-01"]
    y_train = train.passenger_volume_monthly.to_numpy(dtype=float)
    actual = test.passenger_volume_monthly.to_numpy(dtype=float)
    baseline_pred = seasonal_naive(y_train, len(test))
    ets_fit = ExponentialSmoothing(
        y_train, trend="add", seasonal="add", seasonal_periods=12,
        initialization_method="estimated",
    ).fit(optimized=True, remove_bias=True)
    ets_pred = np.asarray(ets_fit.forecast(len(test)))
    baseline_mae = mean_absolute_error(actual, baseline_pred)
    ets_mae = mean_absolute_error(actual, ets_pred)
    ets_wins += int(ets_mae < baseline_mae)
    for period, truth, baseline, ets in zip(test.period, actual, baseline_pred, ets_pred):
        validation_predictions.append({"station_code": code, "period": period, "actual": truth, "seasonal_naive": baseline, "ets": ets})
    station_metrics.append({
        "station_code": code,
        "station_id": group.station_id.iloc[0],
        "seasonal_naive_mae": baseline_mae,
        "seasonal_naive_rmse": mean_squared_error(actual, baseline_pred) ** 0.5,
        "ets_mae": ets_mae,
        "ets_rmse": mean_squared_error(actual, ets_pred) ** 0.5,
    })

validation = pd.DataFrame(validation_predictions)
ets_consistent = ets_wins / df.station_code.nunique() >= 0.60
ets_better_overall = mean_absolute_error(validation.actual, validation.ets) < mean_absolute_error(validation.actual, validation.seasonal_naive)
selected_method = "ets_additive" if ets_consistent and ets_better_overall else "seasonal_naive"
selected_column = "ets" if selected_method == "ets_additive" else "seasonal_naive"

fitted_models, result_rows, clamp_count = {}, [], 0
for code, group in df.groupby("station_code", sort=True):
    metadata = group.iloc[0]
    values = group.passenger_volume_monthly.to_numpy(dtype=float)
    for row in group.itertuples():
        result_rows.append({
            "station_code": row.station_code, "station_id": row.station_id,
            "station_name": row.station_name, "line": row.line, "period": row.period,
            "record_type": "actual", "actual_passengers": int(row.passenger_volume_monthly),
            "predicted_passengers": None, "lower_80": None, "upper_80": None,
            "method": None, "source_status": "synthetic_prototype",
        })
    if selected_method == "ets_additive":
        fit = ExponentialSmoothing(values, trend="add", seasonal="add", seasonal_periods=12, initialization_method="estimated").fit(optimized=True, remove_bias=True)
        prediction = np.asarray(fit.forecast(3))
        fitted_models[code] = fit
    else:
        prediction = seasonal_naive(values, 3)
        fitted_models[code] = {"history": values[-12:].tolist()}
    residual = validation.loc[validation.station_code == code, "actual"] - validation.loc[validation.station_code == code, selected_column]
    sigma = float(np.sqrt(np.mean(np.square(residual))))
    future_periods = pd.date_range(group.period_date.max() + pd.offsets.MonthBegin(1), periods=3, freq="MS")
    for period, raw_prediction in zip(future_periods, prediction):
        clamp_count += int(raw_prediction < 0)
        point = max(0.0, float(raw_prediction))
        result_rows.append({
            "station_code": code, "station_id": metadata.station_id,
            "station_name": metadata.station_name, "line": metadata.line,
            "period": period.strftime("%Y-%m"), "record_type": "forecast",
            "actual_passengers": None, "predicted_passengers": round(point),
            "lower_80": round(max(0, point - 1.282 * sigma)), "upper_80": round(point + 1.282 * sigma),
            "method": selected_method, "source_status": "synthetic_prototype",
        })

bundle = {
    "model_type": selected_method, "models_by_station_code": fitted_models,
    "horizon_months": 3, "trained_through": df.period.max(),
    "station_crosswalk": crosswalk[["station_code", "station_id"]].to_dict("records"),
    "source_status": "synthetic_prototype",
}
with (models_dir / "forecasting_bundle.pkl").open("wb") as file:
    pickle.dump(bundle, file)
pd.DataFrame(result_rows).to_csv(outputs_dir / "station_forecasts.csv", index=False)

overall_actual = validation.actual.to_numpy()
overall_prediction = validation[selected_column].to_numpy()
metrics = {
    "selected_method": selected_method,
    "selection_rule": "ETS requires lower aggregate MAE and wins at least 60% of stations; otherwise seasonal-naive",
    "ets_station_win_rate": ets_wins / df.station_code.nunique(),
    "holdout": {"train": "2023-01..2024-12", "validation": "2025-01..2025-12"},
    "overall": {
        "mae": mean_absolute_error(overall_actual, overall_prediction),
        "rmse": mean_squared_error(overall_actual, overall_prediction) ** 0.5,
        "mae_pct_of_mean_actual": mean_absolute_error(overall_actual, overall_prediction) / overall_actual.mean() * 100,
        "rmse_pct_of_mean_actual": mean_squared_error(overall_actual, overall_prediction) ** 0.5 / overall_actual.mean() * 100,
    },
    "negative_predictions_clamped": clamp_count,
    "per_station": station_metrics,
    "source_status": "synthetic_prototype",
}
(outputs_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
schema = {"target": "passenger_volume_monthly", "identifier": ["station_code", "station_id"], "time_field": "period", "allowed_features": ["historical target values", "month/seasonality"], "horizon_months": 3}
(outputs_dir / "feature_schema.json").write_text(json.dumps(schema, indent=2), encoding="utf-8")
manifest = {"run_at_utc": datetime.now(timezone.utc).isoformat(), "source": str(data_path.relative_to(ROOT)), "records": len(df), "period_range": [df.period.min(), df.period.max()], "model": selected_method, "seed": SEED, "python": platform.python_version(), "sklearn": sklearn.__version__, "source_status": "synthetic_prototype", "limitations": "Synthetic 36-month history; forecasts are prototype-only."}
(outputs_dir / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps({"quality_passed": quality_passed, "selected_method": selected_method, "forecast_rows": len(result_rows), "model": str(models_dir / 'forecasting_bundle.pkl')}, indent=2))


{
  "quality_passed": true,
  "selected_method": "ets_additive",
  "forecast_rows": 3003,
  "model": "C:\\Users\\axels\\Axel Documents\\Documents\\BINUS\\Lomba\\MAPID WebGIS (Top 50)\\App\\TCI\\ml\\forecasting\\models\\forecasting_bundle.pkl"
}
